# Bronze to Silver

In [0]:
%run ./00_setup_and_logging

# Setup and logging

Using workspace.dav_project


## Bronze tables

ok: bronze_commits
ok: bronze_issues
ok: bronze_pulls


ok: bronze_quarantine


...

ok: silver_commit, silver_issue, silver_pull_request, silver_quarantine


## Log table

ok: pipeline_execution_logs


## Small helper functions

helpers ready: new_run_id, now_utc, log_run, get_merge_metrics


In [0]:
dbutils.widgets.text("batch_id", "", "Batch ID (blank = process every Bronze row)")
BATCH_ID = dbutils.widgets.get("batch_id").strip() or None
RUN_ID = new_run_id()
print("batch_id filter:", BATCH_ID or "(none, processing all of Bronze)")
print("run_id:", RUN_ID)

batch_id filter: (none, processing all of Bronze)
run_id: 00528f04-fd07-41ee-8fdc-998d88496f7d


## A salt for hashing names and emails

In [0]:
PII_SALT = "dav-project-2026-fixed-salt-change-if-you-want-but-keep-it-the-same"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, IntegerType,
    BooleanType, ArrayType,
)

## Explicit schemas for the JSON text sitting inside Bronze

In [0]:
COMMIT_JSON_SCHEMA = StructType([
    StructField("sha", StringType()),
    StructField("commit", StructType([
        StructField("message", StringType()),
        StructField("author", StructType([
            StructField("name", StringType()),
            StructField("email", StringType()),
            StructField("date", StringType()),
        ])),
        StructField("committer", StructType([
            StructField("name", StringType()),
            StructField("email", StringType()),
            StructField("date", StringType()),
        ])),
    ])),
    StructField("author", StructType([StructField("login", StringType())])),
    StructField("committer", StructType([StructField("login", StringType())])),
    StructField("parents", ArrayType(StructType([StructField("sha", StringType())]))),
])

ISSUE_JSON_SCHEMA = StructType([
    StructField("id", LongType()),
    StructField("number", IntegerType()),
    StructField("state", StringType()),
    StructField("user", StructType([StructField("login", StringType())])),
    StructField("created_at", StringType()),
    StructField("updated_at", StringType()),
    StructField("closed_at", StringType()),
    StructField("comments", IntegerType()),
    StructField("pull_request", StructType([StructField("url", StringType())])),
])

PULL_JSON_SCHEMA = StructType([
    StructField("id", LongType()),
    StructField("number", IntegerType()),
    StructField("state", StringType()),
    StructField("user", StructType([StructField("login", StringType())])),
    StructField("created_at", StringType()),
    StructField("updated_at", StringType()),
    StructField("closed_at", StringType()),
    StructField("merged_at", StringType()),
    StructField("draft", BooleanType()),
    StructField("base", StructType([StructField("ref", StringType())])),
])

TS_FMT = "yyyy-MM-dd'T'HH:mm:ss'Z'"


def hash_key(*cols):
    """sha2 of the salt plus the first non-null value given. Pseudonymous, stable."""
    picked = F.coalesce(*cols, F.lit("unknown"))
    return F.sha2(F.concat(F.lit(PII_SALT), picked), 256)

## Commits: Bronze -> Silver

In [0]:
def silver_commits():
    start = now_utc()
    try:
        bronze = spark.table(f"{CATALOG}.{SCHEMA}.bronze_commits")
        if BATCH_ID:
            bronze = bronze.filter(F.col("batch_id") == BATCH_ID)

        parsed = bronze.withColumn("data", F.from_json(F.col("raw_json"), COMMIT_JSON_SCHEMA))

        bad = parsed.filter(F.col("data").isNull() | F.col("data.sha").isNull())
        good = parsed.filter(F.col("data").isNotNull() & F.col("data.sha").isNotNull())

        silver = good.select(
            F.col("data.sha").alias("sha"),
            F.col("source_repo").alias("repo"),
            hash_key(F.col("data.author.login"), F.col("data.commit.author.email")).alias("author_key"),
            hash_key(F.col("data.committer.login"), F.col("data.commit.committer.email")).alias("committer_key"),
            F.to_timestamp(F.col("data.commit.author.date"), TS_FMT).alias("committed_at"),
            F.length(F.col("data.commit.message")).alias("message_length"),
            (F.size(F.coalesce(F.col("data.parents"), F.array())) > 1).alias("is_merge_commit"),
            F.lit(now_utc()).alias("load_timestamp"),
        ).dropDuplicates(["sha"])

        inserted = updated = 0
        if silver.limit(1).count() > 0:
            silver.createOrReplaceTempView("silver_commits_batch")
            full_table = f"{CATALOG}.{SCHEMA}.silver_commit"
            spark.sql(f"""
                MERGE INTO {full_table} AS t
                USING silver_commits_batch AS s
                ON t.sha = s.sha
                WHEN MATCHED THEN UPDATE SET *
                WHEN NOT MATCHED THEN INSERT *
            """)
            m = get_merge_metrics(full_table)
            inserted, updated = m["inserted"], m["updated"]

        bad_count = bad.count()
        if bad_count > 0:
            bad.select(
                F.lit("commit").alias("entity_type"),
                F.coalesce(F.col("record_key"), F.lit("unknown")).alias("record_key"),
                F.col("source_repo").alias("repo"),
                F.lit("could not parse commit JSON or missing sha").alias("reason"),
                F.col("raw_json"),
                F.col("batch_id"),
                F.lit(RUN_ID).alias("run_id"),
                F.lit(now_utc()).alias("load_timestamp"),
            ).write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.silver_quarantine")

        log_run(RUN_ID, "Bronze-to-Silver", "commits", {"batch_id": BATCH_ID},
                start, now_utc(), "SUCCESS", inserted, updated, bad_count)
        return inserted, updated, bad_count
    except Exception as e:
        log_run(RUN_ID, "Bronze-to-Silver", "commits", {"batch_id": BATCH_ID},
                start, now_utc(), "FAILED", error_message=str(e))
        print("FAILED: commits:", e)
        return 0, 0, 0

## Issues: Bronze -> Silver

In [0]:
def silver_issues():
    start = now_utc()
    try:
        bronze = spark.table(f"{CATALOG}.{SCHEMA}.bronze_issues")
        if BATCH_ID:
            bronze = bronze.filter(F.col("batch_id") == BATCH_ID)

        parsed = bronze.withColumn("data", F.from_json(F.col("raw_json"), ISSUE_JSON_SCHEMA))

        is_real_issue = F.col("data.pull_request").isNull()
        bad = parsed.filter(F.col("data").isNull() | F.col("data.id").isNull())
        good = parsed.filter(F.col("data").isNotNull() & F.col("data.id").isNotNull() & is_real_issue)

        closed_at_ts = F.to_timestamp(F.col("data.closed_at"), TS_FMT)
        created_at_ts = F.to_timestamp(F.col("data.created_at"), TS_FMT)

        silver = good.select(
            F.col("data.id").cast("string").alias("issue_id"),
            F.col("source_repo").alias("repo"),
            F.col("data.number").alias("number"),
            F.col("data.state").alias("state"),
            hash_key(F.col("data.user.login")).alias("author_key"),
            created_at_ts.alias("created_at"),
            F.to_timestamp(F.col("data.updated_at"), TS_FMT).alias("updated_at"),
            closed_at_ts.alias("closed_at"),
            F.col("data.comments").alias("comment_count"),
            F.when(closed_at_ts.isNotNull(),
                   (F.unix_timestamp(closed_at_ts) - F.unix_timestamp(created_at_ts)) / 3600.0
                   ).alias("time_to_close_hours"),
            F.lit(now_utc()).alias("load_timestamp"),
        ).dropDuplicates(["issue_id"])

        inserted = updated = 0
        if silver.limit(1).count() > 0:
            silver.createOrReplaceTempView("silver_issues_batch")
            full_table = f"{CATALOG}.{SCHEMA}.silver_issue"
            spark.sql(f"""
                MERGE INTO {full_table} AS t
                USING silver_issues_batch AS s
                ON t.issue_id = s.issue_id
                WHEN MATCHED THEN UPDATE SET *
                WHEN NOT MATCHED THEN INSERT *
            """)
            m = get_merge_metrics(full_table)
            inserted, updated = m["inserted"], m["updated"]

        bad_count = bad.count()
        if bad_count > 0:
            bad.select(
                F.lit("issue").alias("entity_type"),
                F.coalesce(F.col("record_key"), F.lit("unknown")).alias("record_key"),
                F.col("source_repo").alias("repo"),
                F.lit("could not parse issue JSON or missing id").alias("reason"),
                F.col("raw_json"),
                F.col("batch_id"),
                F.lit(RUN_ID).alias("run_id"),
                F.lit(now_utc()).alias("load_timestamp"),
            ).write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.silver_quarantine")

        log_run(RUN_ID, "Bronze-to-Silver", "issues", {"batch_id": BATCH_ID},
                start, now_utc(), "SUCCESS", inserted, updated, bad_count)
        return inserted, updated, bad_count
    except Exception as e:
        log_run(RUN_ID, "Bronze-to-Silver", "issues", {"batch_id": BATCH_ID},
                start, now_utc(), "FAILED", error_message=str(e))
        print("FAILED: issues:", e)
        return 0, 0, 0

## Pull requests: Bronze -> Silver

In [0]:
def silver_pulls():
    start = now_utc()
    try:
        bronze = spark.table(f"{CATALOG}.{SCHEMA}.bronze_pulls")
        if BATCH_ID:
            bronze = bronze.filter(F.col("batch_id") == BATCH_ID)

        parsed = bronze.withColumn("data", F.from_json(F.col("raw_json"), PULL_JSON_SCHEMA))

        bad = parsed.filter(F.col("data").isNull() | F.col("data.id").isNull())
        good = parsed.filter(F.col("data").isNotNull() & F.col("data.id").isNotNull())

        created_at_ts = F.to_timestamp(F.col("data.created_at"), TS_FMT)
        merged_at_ts = F.to_timestamp(F.col("data.merged_at"), TS_FMT)

        silver = good.select(
            F.col("data.id").cast("string").alias("pr_id"),
            F.col("source_repo").alias("repo"),
            F.col("data.number").alias("number"),
            F.col("data.state").alias("state"),
            hash_key(F.col("data.user.login")).alias("author_key"),
            created_at_ts.alias("created_at"),
            F.to_timestamp(F.col("data.updated_at"), TS_FMT).alias("updated_at"),
            F.to_timestamp(F.col("data.closed_at"), TS_FMT).alias("closed_at"),
            merged_at_ts.alias("merged_at"),
            merged_at_ts.isNotNull().alias("is_merged"),
            F.coalesce(F.col("data.draft"), F.lit(False)).alias("is_draft"),
            F.col("data.base.ref").alias("base_branch"),
            F.when(merged_at_ts.isNotNull(),
                   (F.unix_timestamp(merged_at_ts) - F.unix_timestamp(created_at_ts)) / 3600.0
                   ).alias("time_to_merge_hours"),
            F.lit(now_utc()).alias("load_timestamp"),
        ).dropDuplicates(["pr_id"])

        inserted = updated = 0
        if silver.limit(1).count() > 0:
            silver.createOrReplaceTempView("silver_pulls_batch")
            full_table = f"{CATALOG}.{SCHEMA}.silver_pull_request"
            spark.sql(f"""
                MERGE INTO {full_table} AS t
                USING silver_pulls_batch AS s
                ON t.pr_id = s.pr_id
                WHEN MATCHED THEN UPDATE SET *
                WHEN NOT MATCHED THEN INSERT *
            """)
            m = get_merge_metrics(full_table)
            inserted, updated = m["inserted"], m["updated"]

        bad_count = bad.count()
        if bad_count > 0:
            bad.select(
                F.lit("pull_request").alias("entity_type"),
                F.coalesce(F.col("record_key"), F.lit("unknown")).alias("record_key"),
                F.col("source_repo").alias("repo"),
                F.lit("could not parse pull request JSON or missing id").alias("reason"),
                F.col("raw_json"),
                F.col("batch_id"),
                F.lit(RUN_ID).alias("run_id"),
                F.lit(now_utc()).alias("load_timestamp"),
            ).write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.silver_quarantine")

        log_run(RUN_ID, "Bronze-to-Silver", "pulls", {"batch_id": BATCH_ID},
                start, now_utc(), "SUCCESS", inserted, updated, bad_count)
        return inserted, updated, bad_count
    except Exception as e:
        log_run(RUN_ID, "Bronze-to-Silver", "pulls", {"batch_id": BATCH_ID},
                start, now_utc(), "FAILED", error_message=str(e))
        print("FAILED: pulls:", e)
        return 0, 0, 0

In [0]:
print("commits:", silver_commits())
print("issues:", silver_issues())
print("pulls:", silver_pulls())

[log] Bronze-to-Silver / commits -> SUCCESS (inserted=0, updated=40, quarantined=0)
commits: (0, 40, 0)
[log] Bronze-to-Silver / issues -> SUCCESS (inserted=0, updated=86, quarantined=0)
issues: (0, 86, 0)
[log] Bronze-to-Silver / pulls -> SUCCESS (inserted=1, updated=225, quarantined=0)
pulls: (1, 225, 0)


In [0]:
display(spark.sql(f"""
    SELECT run_id, layer, entity, status, rows_inserted, rows_updated, rows_quarantined, start_time
    FROM {CATALOG}.{SCHEMA}.pipeline_execution_logs
    WHERE run_id = '{RUN_ID}'
    ORDER BY start_time
"""))

run_id,layer,entity,status,rows_inserted,rows_updated,rows_quarantined,start_time
00528f04-fd07-41ee-8fdc-998d88496f7d,Bronze-to-Silver,commits,SUCCESS,0,40,0,2026-10-11T10:44:21.534Z
00528f04-fd07-41ee-8fdc-998d88496f7d,Bronze-to-Silver,issues,SUCCESS,0,86,0,2026-10-11T10:44:31.780Z
00528f04-fd07-41ee-8fdc-998d88496f7d,Bronze-to-Silver,pulls,SUCCESS,1,225,0,2026-10-11T10:44:40.207Z
